# Warm up
Let's start with some warm up exercises to get you familiar with the database and do some SQL querying.
Make sure to have downloaded the NBA dataset form Kaggle as described in the README.

## How the data loader works

The block `if __name__ == "__main__": main()` calls `main()` when
`data_loader.py` is run directly. It does not call `main()` when the module
is imported, so importing it does not automatically start the download.

Click provides the command-line interface. `@click.command()` makes `main`
a command, and `@click.option()` defines its options: `-d`/`--dataset` is
required, while `-u`/`--user-name` and `-k`/`--api-key` are optional.
When `main()` is called, Click parses these options, checks that required
options are present, and passes their values to the function. It also
provides help through `--help`.


In [ ]:
import sqlite3 as sql
import pandas as pd
from pathlib import Path

## TODO: Use pathlib to get the path to the data directory. Path(__file__) won't work in a Jupyter notebook.
DATA_PATH = Path.cwd() / 'data'

# SQLite needs the database file, rather than the directory containing it.
DATA_PATH = DATA_PATH / "nba.sqlite"


## Create a SQLite connection and run some SQL queries

In [ ]:
con = sql.connect(DATA_PATH)  # connect to the database
query = 'SELECT * FROM game LIMIT 10'  # write a query that selects the first 10 rows from the game table
top_10_game = pd.read_sql(query, con)  # read the query into a pandas dataframe
top_10_game.head()  # print the dataframe

In [ ]:
# let's look at the schema of the database
# TODO: sqlite_master is the name of the table you want to inspeact: Replace TABLE_NAME with the name of the table you want to inspect
query = """
    SELECT *
    -- FILL OUT TABLE NAME
    FROM sqlite_master
"""

pd.read_sql(query, con).sort_values('name')

In [ ]:
# let's see all the tables we have in the database
# TODO: Filter for tables only
query = """
    SELECT name
    FROM sqlite_master
    -- FILL OUT FILTER CONDITION
    WHERE type = 'table'
    ORDER BY name
"""
pd.read_sql(query, con)

In [ ]:
# let's have a look at the columns in the game table
query = """
    PRAGMA table_info(draft_combine_stats)
"""
# Inspect game, as requested above; the supplied example is preserved.
query = "PRAGMA table_info(game)"
pd.read_sql(query, con)

In [ ]:
# let's find the earliest recorded game
query = """
    SELECT game_date
    FROM game
    ORDER BY game_date
    LIMIT 1
"""

print(pd.read_sql(query, con))

# let's find the latest recorded game
query = """
    -- FILL OUT
    SELECT game_date
    FROM game
    WHERE game_date IS NOT NULL
    ORDER BY game_date DESC
    LIMIT 1
"""

print(pd.read_sql(query, con))


In [ ]:
# TODO: Find four interesting facts about the data and write a query to find the answer

# Fact 1: Which five regular-season games had the highest combined scores?
# Keep the season type visible and restrict comparisons to regular-season games.
query = """
    SELECT game_date, team_name_home, team_name_away,
           pts_home, pts_away, pts_home + pts_away AS total_points
    FROM game
    WHERE season_type = 'Regular Season'
      AND pts_home IS NOT NULL AND pts_away IS NOT NULL
    ORDER BY total_points DESC, game_date, game_id
    LIMIT 5
"""
highest_scoring_games = pd.read_sql(query, con)
highest_scoring_games


In [ ]:
# Fact 2: Which five regular-season games had the largest scoring margins?
query = """
    SELECT game_date, team_name_home, team_name_away,
           pts_home, pts_away, ABS(pts_home - pts_away) AS point_margin
    FROM game
    WHERE season_type = 'Regular Season'
      AND pts_home IS NOT NULL AND pts_away IS NOT NULL
    ORDER BY point_margin DESC, game_date, game_id
    LIMIT 5
"""
largest_margins = pd.read_sql(query, con)
largest_margins


In [ ]:
# Fact 3: What percentage of recorded regular-season games did the home team win?
# Use only games with a recorded win/loss outcome for the denominator.
query = """
    SELECT COUNT(*) AS games_with_result,
           SUM(CASE WHEN wl_home = 'W' THEN 1 ELSE 0 END) AS home_wins,
           ROUND(100.0 * AVG(CASE WHEN wl_home = 'W' THEN 1.0 ELSE 0.0 END), 2)
               AS home_win_percent
    FROM game
    WHERE season_type = 'Regular Season' AND wl_home IN ('W', 'L')
"""
home_win_rate = pd.read_sql(query, con)
home_win_rate


In [ ]:
# Fact 4: Longest winning and losing streaks in recorded regular-season games.
# Streaks continue across seasons; playoffs and preseason are excluded.
# Track franchises by team ID, so a change of team name does not reset a streak.
# Consecutive identical results form a group; rank the groups by their length.
# Include ties and show the dates and team names recorded during each streak.
query = """
WITH team_games AS (
    SELECT team_id_home AS team_id, team_name_home AS team_name,
           game_id, game_date, wl_home AS result
    FROM game
    WHERE season_type = 'Regular Season'
    UNION ALL
    SELECT team_id_away, team_name_away, game_id, game_date, wl_away
    FROM game
    WHERE season_type = 'Regular Season'
), numbered AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY team_id ORDER BY game_date, game_id
           ) - ROW_NUMBER() OVER (
               PARTITION BY team_id, result ORDER BY game_date, game_id
           ) AS streak_group
    FROM team_games
), streaks AS (
    SELECT team_id, result, streak_group,
           GROUP_CONCAT(DISTINCT team_name) AS team_names,
           COUNT(*) AS games,
           DATE(MIN(game_date)) AS start_date,
           DATE(MAX(game_date)) AS end_date
    FROM numbered
    GROUP BY team_id, result, streak_group
), ranked AS (
    SELECT *, RANK() OVER (
        PARTITION BY result ORDER BY games DESC
    ) AS position
    FROM streaks
)
SELECT CASE result WHEN 'W' THEN 'Winning' ELSE 'Losing' END AS streak,
       team_id, team_names, games, start_date, end_date
FROM ranked
WHERE position = 1
ORDER BY CASE result WHEN 'W' THEN 0 ELSE 1 END, team_names
"""

longest_streaks = pd.read_sql(query, con)
longest_streaks
